# Result · The ceiling reference changes measured hackability

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** the Sec. V paragraph "The reference of the hack rule changes the answer" and the factor-of-four claim of the abstract.

Every E1 statistic under both success rules: score gain over the raw input x (as originally run) and over the codec
reconstruction x̂ (corrected), with the paired per-seed differences against R0. Recomputes E1 (24 jobs, ~15 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1']]
print(len(KEYS), "runs:", ", ".join(KEYS))

24 runs: e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e1_R3ms_s0, e1_R3ms_s1, e1_R3ms_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    24
Name: count, dtype: int64

In [6]:
display(Markdown((OUT / "e1_both_refs.md").read_text()))

# E1 under both ceiling references

tau=0.3; panel margins {'estoi': 0.02, 'sisdr': 1.0, 'wer': 0.0, 'sbs': 0.02}; mean over seeds; mean-action candidate only.

## Predictor's own reconstruction offset R(x_hat)-R(x)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.072 |         -0.095 |          0 |           -0.095 |               -0.095 |
| R1              |      0.074 |         -0.064 |          0 |           -0.064 |               -0.064 |
| R2              |      0.081 |          0.001 |          0 |            0.001 |                0.001 |
| R3              |      0.081 |         -0.266 |          0 |           -0.266 |               -0.266 |
| R3ms            |      0.04  |          0.949 |          0 |            0.949 |                0.949 |
| baseline_noise  |      0.069 |         -0.085 |          0 |           -0.085 |               -0.085 |
| baseline_pgd    |      0.082 |         -0.077 |          0 |           -0.077 |               -0.077 |
| baseline_random |      0.09  |         -0.03  |          0 |           -0.03  |               -0.03  |

## ASR, ceiling referenced to raw input x (as run)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.12  |          0.181 |      0.999 |            0.012 |                0.251 |
| R1              |      0.301 |          0.067 |      1     |            0.001 |                0     |
| R2              |      0.304 |          0.079 |      1     |            0     |                0     |
| R3              |      0.293 |          0.05  |      1     |            0     |                0     |
| R3ms            |      0.037 |          0.306 |      0.841 |            0.022 |                0     |
| baseline_noise  |      0.356 |          0.235 |      1     |            0.013 |                0.217 |
| baseline_pgd    |      0.352 |          0.21  |      0.969 |            0.01  |                0.181 |
| baseline_random |      0.35  |          0.053 |      1     |            0     |                0.002 |

## ASR, ceiling referenced to codec reconstruction x_hat (corrected)

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |      0.065 |          0.226 |      0.999 |            0.001 |                0.318 |
| R1              |      0.161 |          0.05  |      1     |            0     |                0     |
| R2              |      0.141 |          0.017 |      1     |            0     |                0     |
| R3              |      0.127 |          0.137 |      1     |            0.002 |                0     |
| R3ms            |      0.016 |          0.082 |      0.841 |            0     |                0     |
| baseline_noise  |      0.23  |          0.27  |      1     |            0     |                0.246 |
| baseline_pgd    |      0.225 |          0.223 |      0.969 |            0     |                0.18  |
| baseline_random |      0.2   |          0.019 |      1     |            0     |                0     |

## Attacker gain over x_hat

| target          |   grpo_dac |   grpo_encodec |   pgd_wave |   random_encodec |   transfer_r0_round1 |
|:----------------|-----------:|---------------:|-----------:|-----------------:|---------------------:|
| R0              |     -0.051 |          0.136 |      1.786 |           -0.344 |                0.245 |
| R1              |      0.142 |          0.138 |      1.935 |           -0.913 |               -1.56  |
| R2              |      0.134 |          0.03  |      2.003 |           -1.303 |               -1.743 |
| R3              |      0.122 |          0.216 |      2.102 |           -1.614 |               -2.491 |
| R3ms            |      0.048 |         -0.455 |      1.227 |           -2.713 |               -3.51  |
| baseline_noise  |      0.171 |          0.223 |      2.04  |           -0.388 |                0.191 |
| baseline_pgd    |      0.176 |          0.205 |      1.618 |           -0.684 |                0.078 |
| baseline_random |      0.156 |          0.102 |      2.055 |           -1.887 |               -0.959 |


In [7]:
print((OUT / "e1_reference_check.md").read_text())

asr_ref_input: ASR(R0)-ASR(R3) per seed [0.143, 0.005, 0.247], mean +0.131, bootstrap 95% CI [+0.005, +0.247]
asr_ref_input: ASR(R0)-ASR(R3ms) per seed [0.083, -0.729, 0.272], mean -0.125, bootstrap 95% CI [-0.729, +0.272]
asr_ref_input: ASR(R0)-ASR(baseline_random) per seed [0.119, -0.017, 0.282], mean +0.128, bootstrap 95% CI [-0.017, +0.282]
asr_ref_recon: ASR(R0)-ASR(R3) per seed [-0.073, -0.039, 0.379], mean +0.089, bootstrap 95% CI [-0.073, +0.379]
asr_ref_recon: ASR(R0)-ASR(R3ms) per seed [0.227, -0.2, 0.404], mean +0.144, bootstrap 95% CI [-0.200, +0.404]
asr_ref_recon: ASR(R0)-ASR(baseline_random) per seed [0.234, -0.007, 0.394], mean +0.207, bootstrap 95% CI [-0.007, +0.394]


In [8]:
repro.compare_csv(OUT / "e1_both_refs.csv", COMMITTED_OUT / "e1_both_refs.csv")

seed                       0.0
n_utt                      0.0
recon_gain                 0.0
asr_ref_input              0.0
asr_ref_recon              0.0
ceiling_ref_input          0.0
ceiling_ref_recon          0.0
panel_rate                 0.0
gain_ref_input             0.0
gain_ref_recon             0.0
estoi_delta                0.0
sisdr_delta                0.0
sbs_delta                  0.0
wer_delta                  0.0
judge_dnsmos_sig_delta     0.0
judge_dnsmos_bak_delta     0.0
judge_dnsmos_ovrl_delta    0.0
judge_dnsmos_p808_delta    0.0
judge_utmos_delta          0.0
Name: max_abs_diff, dtype: float64

In [9]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('e1_',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

1320 numbers; max |diff| = 0.0000; identical: 1320


,paper,reproduced,abs_diff
e1_R0_grpo_dac_asr_ref_input_2dp,0.120,0.120,0.0
e1_R0_grpo_dac_asr_ref_input_3dp,0.120,0.120,0.0
e1_R0_grpo_dac_asr_ref_input_sd,0.192,0.192,0.0
e1_R0_grpo_dac_asr_ref_recon_2dp,0.070,0.070,0.0
e1_R0_grpo_dac_asr_ref_recon_3dp,0.065,0.065,0.0
e1_R0_grpo_dac_asr_ref_recon_sd,0.113,0.113,0.0
e1_R0_grpo_dac_ceiling_ref_recon_2dp,0.070,0.070,0.0
e1_R0_grpo_dac_ceiling_ref_recon_3dp,0.066,0.066,0.0
e1_R0_grpo_dac_ceiling_ref_recon_sd,0.114,0.114,0.0
e1_R0_grpo_dac_estoi_delta_2dp,-0.010,-0.010,0.0
